# NYC Airbnb Room Type Prediction

This notebook documents a room-type classification experiment using the supplied 2019 New York City Airbnb listings. It compares two input scenarios: one without review history and one with review history. The final holdout separates hosts between training and test data.

The goal is to understand where the models work, where they fail, and what remains before deployment. The exported models are candidates, not a claim of production approval.

In [1]:
# Project setup for Google Colab and local Jupyter
from pathlib import Path
import os
import sys
import zipfile

REQUIRED_FILES = [
    "AB_NYC_2019.csv",
    "inference.py",
    "heldout_metrics.csv",
    "classification_reports.json",
    "confusion_matrices.json",
    "calibration_diagnostics.csv",
    "subgroup_diagnostics.csv",
    "permutation_importance.csv",
    "monitoring.py",
    "artifacts/airbnb_room_type_without_reviews.joblib",
    "artifacts/airbnb_room_type_with_reviews.joblib",
]

def is_project_root(path):
    path = Path(path)
    return path.is_dir() and all((path / name).is_file() for name in REQUIRED_FILES)

# First check common locations. This also supports running from the extracted ZIP folder.
_candidates = [
    Path.cwd(),
    Path.cwd() / "airbnb_final_release",
    Path("/content/nyc_airbnb_project"),
    Path("/content/airbnb_final_release"),
]
ROOT = next((p.resolve() for p in _candidates if is_project_root(p)), None)

if ROOT is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "Project companion files were not found. In Google Colab, upload "
            "NYC_Airbnb_Colab_Ready.zip when prompted. In local Jupyter, extract "
            "the ZIP and open this notebook from the extracted project folder."
        ) from exc

    print("The notebook needs its companion files. Please upload NYC_Airbnb_Colab_Ready.zip.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise ValueError("Please upload the complete project ZIP, not just the notebook or CSV.")

    extract_dir = Path("/content/nyc_airbnb_project").resolve()
    extract_dir.mkdir(parents=True, exist_ok=True)
    for zip_name in zip_names:
        with zipfile.ZipFile(zip_name) as archive:
            # Reject path traversal entries before extraction.
            for member in archive.infolist():
                destination = (extract_dir / member.filename).resolve()
                if destination != extract_dir and extract_dir not in destination.parents:
                    raise ValueError(f"Unsafe path in project archive: {member.filename}")
            archive.extractall(extract_dir)

    possible_roots = [p.parent for p in extract_dir.rglob("AB_NYC_2019.csv")]
    ROOT = next((p.resolve() for p in possible_roots if is_project_root(p)), None)
    if ROOT is None:
        raise FileNotFoundError(
            "The uploaded ZIP does not contain the complete project structure. "
            "Please use NYC_Airbnb_Colab_Ready.zip."
        )

missing = [name for name in REQUIRED_FILES if not (ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Project bundle is incomplete. Missing files: {missing}")

# Make companion Python modules importable and make relative paths predictable.
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


from inference import load_model, predict

print(f"Project files found: {ROOT}")
print("Dataset, inference helper, diagnostic files, and both model artifacts are available.")


Project files found: /mnt/data/colab_debug/extracted/airbnb_final_release
Dataset, inference helper, diagnostic files, and both model artifacts are available.


## Load the data and inspect its quality



In [2]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv(ROOT / 'AB_NYC_2019.csv')
print(f'{len(df):,} listings and {df.shape[1]} columns')
display(df.head())
quality = pd.DataFrame({'missing': df.isna().sum(), 'missing_percent': (df.isna().mean() * 100).round(2)}).sort_values('missing', ascending=False)
display(quality)
print('Exact duplicate rows:', int(df.duplicated().sum()))
print('Unique listing IDs:', int(df['id'].nunique()))
print('Unique hosts:', int(df['host_id'].nunique()))
assert df['id'].is_unique and not df.duplicated().any()
assert (df['reviews_per_month'].isna() == df['number_of_reviews'].eq(0)).all()

48,895 listings and 16 columns


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


,missing,missing_percent
last_review,10052,20.56
reviews_per_month,10052,20.56
host_name,21,0.04
name,16,0.03
neighbourhood_group,0,0.00
neighbourhood,0,0.00
id,0,0.00
host_id,0,0.00
longitude,0,0.00
latitude,0,0.00


Exact duplicate rows: 0
Unique listing IDs: 48895
Unique hosts: 37457




The target has three room types. Shared rooms are a small minority, so overall accuracy alone is not a useful summary of model quality.

In [3]:
target_summary = pd.DataFrame({'listings': df['room_type'].value_counts(), 'share_percent': (df['room_type'].value_counts(normalize=True) * 100).round(2)})
display(target_summary)

,listings,share_percent
room_type,,
Entire home/apt,25409,51.97
Private room,22326,45.66
Shared room,1160,2.37



The first scenario uses listing and location fields but excludes review history. The second adds review count and review rate. A missing review rate is treated as zero only because every such row in this source file has zero reviews.



In [4]:
without_review_features = ['neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'price', 'minimum_nights', 'availability_365', 'calculated_host_listings_count']
with_review_features = without_review_features + ['number_of_reviews', 'reviews_per_month']
excluded = ['id', 'host_id', 'name', 'host_name', 'last_review', 'room_type']
print('Without review history:', without_review_features)
print('With review history:', with_review_features)
print('Excluded from predictors:', excluded)

Without review history: ['neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'price', 'minimum_nights', 'availability_365', 'calculated_host_listings_count']
With review history: ['neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'price', 'minimum_nights', 'availability_365', 'calculated_host_listings_count', 'number_of_reviews', 'reviews_per_month']
Excluded from predictors: ['id', 'host_id', 'name', 'host_name', 'last_review', 'room_type']


In [5]:
X = df[with_review_features]
y = df['room_type']
hosts = df['host_id']
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42).split(X, y, groups=hosts))
train_hosts = set(hosts.iloc[train_idx]); test_hosts = set(hosts.iloc[test_idx])
display(pd.DataFrame([{'training_listings': len(train_idx), 'test_listings': len(test_idx), 'training_hosts': len(train_hosts), 'test_hosts': len(test_hosts), 'host_overlap': len(train_hosts & test_hosts)}]))
assert not (train_hosts & test_hosts)

,training_listings,test_listings,training_hosts,test_hosts,host_overlap
0,34299,14596,26219,11238,0


In [6]:
metrics = pd.read_csv(ROOT / 'heldout_metrics.csv')
display(metrics.set_index('scenario').T)

scenario,without_reviews,with_reviews
model,HistGradientBoostingClassifier(class_weight=ba...,HistGradientBoostingClassifier(class_weight=ba...
test_accuracy,0.852151,0.855851
test_macro_f1,0.702783,0.703837
test_weighted_f1,0.849178,0.853
test_balanced_accuracy,0.67083,0.673553
shared_room_precision,0.61745,0.593548
shared_room_recall,0.288401,0.288401
shared_room_f1,0.393162,0.388186
test_multiclass_brier,0.226373,0.22287
test_log_loss,0.396779,0.390511


In [7]:
reports = json.loads((ROOT / 'classification_reports.json').read_text())
matrices = json.loads((ROOT / 'confusion_matrices.json').read_text())
for scenario in ['without_reviews', 'with_reviews']:
    print('\n', scenario)
    display(pd.DataFrame(reports[scenario]).T)
    display(pd.DataFrame(matrices[scenario], index=['actual: ' + x for x in ['Entire home/apt','Private room','Shared room']], columns=['predicted: ' + x for x in ['Entire home/apt','Private room','Shared room']]))


 without_reviews


,precision,recall,f1-score,support
Entire home/apt,0.863631,0.893969,0.878538,7743.000000
Private room,0.843284,0.830119,0.836650,6534.000000
Shared room,0.617450,0.288401,0.393162,319.000000
accuracy,0.852151,0.852151,0.852151,0.852151
macro avg,0.774788,0.670830,0.702783,14596.000000
weighted avg,0.849142,0.852151,0.849178,14596.000000


,predicted: Entire home/apt,predicted: Private room,predicted: Shared room
actual: Entire home/apt,6922,817,4
actual: Private room,1057,5424,53
actual: Shared room,36,191,92



 with_reviews


,precision,recall,f1-score,support
Entire home/apt,0.869882,0.894485,0.882012,7743.000000
Private room,0.844883,0.837772,0.841313,6534.000000
Shared room,0.593548,0.288401,0.388186,319.000000
accuracy,0.855851,0.855851,0.855851,0.855851
macro avg,0.769438,0.673553,0.703837,14596.000000
weighted avg,0.852652,0.855851,0.853000,14596.000000


,predicted: Entire home/apt,predicted: Private room,predicted: Shared room
actual: Entire home/apt,6926,811,6
actual: Private room,1003,5474,57
actual: Shared room,33,194,92


## Check probability calibration

The model exposes calibrated probabilities, but calibration is never assumed from the existence of `predict_proba`. These diagnostics are calculated on the held-out host-grouped test set.

In [8]:
display(pd.read_csv(ROOT / 'calibration_diagnostics.csv'))

,scenario,class,one_vs_rest_brier,calibration_bins,mean_absolute_bin_gap
0,without_reviews,Entire home/apt,0.099139,10,0.032599
1,without_reviews,Private room,0.110189,10,0.036470
2,without_reviews,Shared room,0.017046,10,0.010045
3,with_reviews,Entire home/apt,0.096865,10,0.035517
4,with_reviews,Private room,0.108698,10,0.035958
5,with_reviews,Shared room,0.017308,10,0.011131


In [9]:
display(pd.read_csv(ROOT / 'subgroup_diagnostics.csv'))

,scenario,dimension,group,n,macro_f1,shared_room_recall
0,without_reviews,borough,Bronx,366,0.536498,0.000000
1,without_reviews,borough,Brooklyn,6002,0.744399,0.395833
2,without_reviews,borough,Manhattan,6456,0.653732,0.183206
3,without_reviews,borough,Queens,1675,0.715800,0.371795
4,without_reviews,borough,Staten Island,97,0.745802,0.500000
5,without_reviews,price_quartile,"(-0.001, 70.0]",3954,0.518621,0.408072
6,without_reviews,price_quartile,"(70.0, 110.0]",3533,0.508532,0.018519
7,without_reviews,price_quartile,"(110.0, 180.0]",3562,0.441044,0.000000
8,without_reviews,price_quartile,"(180.0, 10000.0]",3547,0.379022,0.000000
9,with_reviews,borough,Bronx,366,0.535438,0.000000


In [10]:
importance = pd.read_csv(ROOT / 'permutation_importance.csv')
display(importance.sort_values(['scenario','importance_mean'], ascending=[True,False]))

,scenario,feature,importance_mean,importance_std
12,with_reviews,price,0.411471,0.0
14,with_reviews,availability_365,0.105710,0.0
15,with_reviews,calculated_host_listings_count,0.092412,0.0
11,with_reviews,longitude,0.081362,0.0
10,with_reviews,latitude,0.028196,0.0
17,with_reviews,reviews_per_month,0.022966,0.0
13,with_reviews,minimum_nights,0.022674,0.0
16,with_reviews,number_of_reviews,0.005660,0.0
9,with_reviews,neighbourhood,0.001716,0.0
8,with_reviews,neighbourhood_group,-0.000249,0.0


In [11]:
for scenario, features in [('without_reviews', without_review_features), ('with_reviews', with_review_features)]:
    model = load_model(ROOT / 'artifacts' / f'airbnb_room_type_{scenario}.joblib')
    display(predict(model, df[features].head(3)))

,predicted_room_type,predicted_class_probability,probability_Entire home/apt,probability_Private room,probability_Shared room
0,Private room,0.788194,0.192833,0.788194,0.018973
1,Entire home/apt,0.613117,0.613117,0.381626,0.005256
2,Private room,0.606030,0.356554,0.606030,0.037416


,predicted_room_type,predicted_class_probability,probability_Entire home/apt,probability_Private room,probability_Shared room
0,Private room,0.825716,0.156659,0.825716,0.017625
1,Entire home/apt,0.624074,0.624074,0.372521,0.003405
2,Private room,0.707688,0.260323,0.707688,0.031988


## Check the inference contract

The helper rejects missing required fields, invalid geographic coordinates, negative price/count values, and inconsistent missing review rates. Unknown neighbourhood labels are accepted because the encoder is configured to ignore unseen categories.

In [12]:
import subprocess
import sys

for test_name in ["test_inference.py", "test_monitoring.py"]:
    result = subprocess.run(
        [sys.executable, str(ROOT / test_name)],
        cwd=ROOT,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"{test_name} failed with exit code {result.returncode}")
print("All bundled automated checks passed.")


without_reviews PASS
with_reviews PASS
All inference contract tests passed.



Monitoring utility tests passed.

All bundled automated checks passed.


In [13]:
from monitoring import population_stability_index, retraining_recommendation
print('Example PSI:', population_stability_index([10,12,13,15,18], [11,12,13,16,19]))
print(retraining_recommendation(max_feature_psi=0.04, prediction_psi=0.03))
# Example after downloading a newer labelled NYC snapshot:
# python external_validation.py --csv ./newer_nyc_listings.csv --model artifacts/airbnb_room_type_without_reviews.joblib --snapshot-date 2026-09-14

Example PSI: 14.647213938200336
{'action': 'NO_TRIGGER_FROM_SUPPLIED_METRICS', 'reasons': [], 'automatic_retraining': False, 'note': 'Retrain only after data provenance, labels, validation, calibration, and rollback checks pass.'}
